<img src="https://iteso.mx/documents/27014/202031/Logo-ITESO-MinimoH.png"
     align="right"
     width="300"/>

# Redes Neuronales Convulucionales (CNN) multivariadas para series de tiempo
## *Modelos no lineales para pronósitico*  - Pedro Martinez

---

## Mercado Eléctrico Mayorista en México
Es la "Bolsa de Valores" de la electricidad en México. Nació en 2014 con la Reforma Energética para romper el monopolio de la generación y permitir la libre competencia.
[Video ilustrativo aquí](https://www.enel.mx/es/productos/enel-energia/conoce-mem#:~:text=CENACE:%20Es%20la%20entidad%20encargada%20de%20operar,realmente%20est%C3%A9n%20basados%20en%20costos%20de%20generaci%C3%B3n)

## ¿Qué vamos a predecir?
El Precio de la Energía en el Mercado del Día en Adelanto (MDA) representa el costo marginal de inyectar un Megavatio-hora (MWh) adicional a la red eléctrica con datos del [CENACE](https://www.cenace.gob.mx/CENACE.aspx) y la API del [SIM](https://www.cenace.gob.mx/APSIM.aspx)

Es utilizado en los fondos de inversión y las empresas privadas para jugar y ganar dinero dentro del MEM


## ¿Por qué es importante?
Si una empresa que gestiona sistemas de almacenamiento (baterías a gran escala) sabe a qué hora ocurrirá un pico, puede comprar energía barata en la madrugada, almacenarla y venderla en la hora pico, generando ganancias.


## ¿Que complicaciones existen?
Para cumplir con la demanda de energía a todas horas en todos lados, el CENACE se encarga de organizar a las plantas energéticas públicas y privadas, decidiendo de donde vendrá la energía.


1. Picos de volatilidad:
- Durante las horas de máxima demanda (en la noche o en días mucho calor), las plantas de generación más baratas (solares, eólicas o hidroeléctricas) llegan a su límite de capacidad.
- Para evitar un apagón, el operador del sistema (CENACE) se ve obligado a encender plantas "de pico" —típicamente turbinas de gas o diésel de arranque rápido— que son extremadamente costosas de operar, lo que dispara el precio marginal de toda la red de forma abrupta por un par de horas.

2. Varios puntos geográficos
- El CENACE gestiona más de 2,000 de nodos que forman el Sistema Interconectado Nacional, a estos puntos se les llama subestaciones.
- Hay algunos puntos clave donde se calcula el Precio Marginal Local de la energía a estos se les llama Nodos de Precios (PML).

In [1]:
#@title Librerias
import requests
import pandas as pd
import warnings
from datetime import datetime, timedelta
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import plotly.graph_objects as go
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense


warnings.filterwarnings('ignore')

2026-04-26 11:56:03.888858: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777226163.966486    5060 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777226163.987050    5060 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-26 11:56:04.120009: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
#@title Preprocesamiento para la consulta desde la API

def extraer_precio_cenace(zona, fecha_inicio, fecha_fin, sistema='SIN', proceso='MDA'):
    """
    Extrae el Precio de Energía en Nodos Distribuidos (PEND) del CENACE.
    Maneja automáticamente la paginación de 7 días exigida por la API.
    """
    print(f"Consultando datos para el nodo {zona} ({fecha_inicio} a {fecha_fin})...")

    inicio_dt = datetime.strptime(fecha_inicio, '%Y-%m-%d')
    fin_dt = datetime.strptime(fecha_fin, '%Y-%m-%d')

    dfs_temporales = []
    fecha_actual = inicio_dt

    while fecha_actual <= fin_dt:
        fecha_fin_chunk = min(fecha_actual + timedelta(days=6), fin_dt)
        str_inicio = fecha_actual.strftime('%Y/%m/%d')
        str_fin = fecha_fin_chunk.strftime('%Y/%m/%d')

        url = f"https://ws01.cenace.gob.mx:8082/SWPEND/SIM/{sistema}/{proceso}/{zona}/{str_inicio}/{str_fin}/JSON"

        try:
            response = requests.get(url, verify=False)
            response.raise_for_status()
            data = response.json()

            if data.get('status') == 'OK' and 'Resultados' in data:
                df_chunk = pd.DataFrame(data['Resultados'][0]['Valores'])
                dfs_temporales.append(df_chunk)

        except Exception as e:
            print(f"Error HTTP al consultar el bloque {str_inicio}: {e}")
            return None

        fecha_actual = fecha_fin_chunk + timedelta(days=1)

    if not dfs_temporales:
        print("No se encontraron datos para la consulta solicitada.")
        return None

    # Procesamiento y limpieza del DataFrame final
    df_final = pd.concat(dfs_temporales, ignore_index=True)
    df_final['FechaHora'] = pd.to_datetime(df_final['fecha'] + ' ' + (df_final['hora'].astype(int) - 1).astype(str) + ':00:00')
    df_final = df_final[['FechaHora', 'pz']]
    df_final['pz'] = pd.to_numeric(df_final['pz'])
    df_final.rename(columns={'pz': 'Precio_Zonal_MXN'}, inplace=True)

    df_final.sort_values('FechaHora', inplace=True)
    return df_final.reset_index(drop=True)




In [3]:
#@title Solo correr en caso de que se bloquee el explorador!
import requests
import pandas as pd
import plotly.graph_objects as go
import warnings
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

# ==========================================
# 1. FUNCIÓN DE EXTRACCIÓN (Con parche Anti-Bloqueo)
# ==========================================
def extraer_precio_cenace(zona, fecha_inicio, fecha_fin, sistema='SIN', proceso='MDA'):
    """
    Extrae el Precio de Energía en Nodos Distribuidos (PEND).
    Incluye User-Agent para evitar errores HTTP 403 (Forbidden).
    """
    print(f" Conectando con CENACE para el nodo {zona}...")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    inicio_dt = datetime.strptime(fecha_inicio, '%Y-%m-%d')
    fin_dt = datetime.strptime(fecha_fin, '%Y-%m-%d')

    dfs_temporales = []
    fecha_actual = inicio_dt

    while fecha_actual <= fin_dt:
        fecha_fin_chunk = min(fecha_actual + timedelta(days=6), fin_dt)
        str_inicio = fecha_actual.strftime('%Y/%m/%d')
        str_fin = fecha_fin_chunk.strftime('%Y/%m/%d')

        url = f"https://ws01.cenace.gob.mx:8082/SWPEND/SIM/{sistema}/{proceso}/{zona}/{str_inicio}/{str_fin}/JSON"

        try:
            response = requests.get(url, headers=headers, verify=False, timeout=15)
            response.raise_for_status()
            data = response.json()

            if data.get('status') == 'OK' and 'Resultados' in data:
                df_chunk = pd.DataFrame(data['Resultados'][0]['Valores'])
                dfs_temporales.append(df_chunk)

        except Exception as e:
            print(f"Error en el bloque {str_inicio}: {e}")
            return None

        fecha_actual = fecha_fin_chunk + timedelta(days=1)

    if not dfs_temporales:
        print("No se encontraron datos para la configuración solicitada.")
        return None

    df_final = pd.concat(dfs_temporales, ignore_index=True)
    df_final['FechaHora'] = pd.to_datetime(df_final['fecha'] + ' ' + (df_final['hora'].astype(int) - 1).astype(str) + ':00:00')
    df_final = df_final[['FechaHora', 'pz']]
    df_final['pz'] = pd.to_numeric(df_final['pz'])
    df_final.rename(columns={'pz': 'Precio_Zonal_MXN'}, inplace=True)

    return df_final.sort_values('FechaHora').reset_index(drop=True)

# Fechas
fecha_fin_str = (hoy - timedelta(days=2)).strftime('%Y-%m-%d')
fecha_inicio_str = (hoy - timedelta(days=7)).strftime('%Y-%m-%d')

# Nodos sugeridos: MONTERREY, VDM-CENTRO, VDM-NORTE, MERIDA, HERMOSILLO, TIJUANA (Con Sistema BCA)
nodo_objetivo = 'TOLUCA' #@param {type:"string"}
sistema_electrico = 'SIN' #@param ["SIN", "BCA", "BCS"]

df_precio = extraer_precio_cenace(
    zona=nodo_objetivo,
    fecha_inicio=fecha_inicio_str,
    fecha_fin=fecha_fin_str,
    sistema=sistema_electrico
)

# Visualización
if df_precio is not None:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_precio['FechaHora'],
        y=df_precio['Precio_Zonal_MXN'],
        mode='lines',
        name='Precio MDA',
        line=dict(color='#49A078', width=2)
    ))

    fig.update_layout(
        title=f'<b>Precios de Energía (MDA) - Nodo {nodo_objetivo} ({sistema_electrico})</b>',
        xaxis_title='Fecha',
        yaxis_title='Precio (MXN/MWh)',
        hovermode='x unified',
        font=dict(family="Arial, sans-serif", size=12),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    fig.show()

NameError: name 'hoy' is not defined

In [10]:
#@title Seleccion de datos y visualización
# Seleccionamos las fechas en este formato en especifico y
# cuantos días hacia atras
hoy = datetime.now()
fecha_fin_str = (hoy - timedelta(days=2)).strftime('%Y-%m-%d')
fecha_inicio_str = (hoy - timedelta(days=18)).strftime('%Y-%m-%d')
# Seleccionamos el nodo que estaremos revisando
# MONTERREY, VDM-CENTRO, VDM-NORTE, MERIDA, HERMOSILLO, TIJUANA (Con Sistema BCA)
nodo_objetivo = 'VDM-SUR' #@param {type:"string"} # Lo mas cercano a Toluca que pude encontrar

df_precio = extraer_precio_cenace(
    zona=nodo_objetivo,
    fecha_inicio=fecha_inicio_str,
    fecha_fin=fecha_fin_str,
    sistema='SIN' #@param["SIN", "BCA", "BCS"]
)


# Graficamos
if df_precio is not None:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_precio['FechaHora'],
        y=df_precio['Precio_Zonal_MXN'],
        mode='lines',
        name='Precio MDA',
        line=dict(color='#49A078', width=2)
    ))

    fig.update_layout(
        title=f'<b>Precios de Energía (MDA) - Nodo {nodo_objetivo}</b>',
        xaxis_title='Fecha',
        yaxis_title='Precio (MXN/MWh)',
        hovermode='x unified',
        font=dict(family="Arial, sans-serif", size=12),
        margin=dict(l=40, r=40, t=60, b=40)
    )
    fig.show()

 Conectando con CENACE para el nodo VDM-SUR...


In [11]:
#@title Creación de nueva variable de predicción (Hora) y escalamiento

df_precio['Hora'] = df_precio['FechaHora'].dt.hour
features = ['Precio_Zonal_MXN', 'Hora']
data_raw = df_precio[features].values

WINDOW_SIZE = 24
HORIZON = 1


split_idx = int(len(data_raw) * 0.8)
# Separamos los datos crudos ANTES de escalar
train_data_raw = data_raw[:split_idx]
test_data_raw = data_raw[split_idx - WINDOW_SIZE:]

# Escalamiento
scaler = MinMaxScaler(feature_range=(0, 1))
train_data_scaled = scaler.fit_transform(train_data_raw)
test_data_scaled = scaler.transform(test_data_raw)

# Vemos como quedo
df_precio.head()

,FechaHora,Precio_Zonal_MXN,Hora
0,2026-04-08 00:00:00,467.81,0
1,2026-04-08 01:00:00,444.11,1
2,2026-04-08 02:00:00,439.69,2
3,2026-04-08 03:00:00,438.32,3
4,2026-04-08 04:00:00,429.78,4


In [12]:
#@title Creación de ventanas en 3D


def create_sequences(data, window_size, horizon=1):
    """Transforma la matriz 2D continua en Tensores 3D para Keras."""
    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        # La ventana X toma todas las variables (Precio y Hora)
        X.append(data[i : (i + window_size), :])
        # El objetivo Y es predecir solo el Precio (índice 0)
        y.append(data[i + window_size : i + window_size + horizon, 0])

    return np.array(X), np.array(y)

# Generamos los tensores de forma totalmente independiente
X_train, y_train = create_sequences(train_data_scaled, WINDOW_SIZE, HORIZON)
X_test, y_test = create_sequences(test_data_scaled, WINDOW_SIZE, HORIZON)

print(f"Tensor X_train: {X_train.shape} -> (Muestras, Timesteps, Variables)")
print(f"Tensor X_test : {X_test.shape}")

Tensor X_train: (302, 24, 2) -> (Muestras, Timesteps, Variables)
Tensor X_test : (82, 24, 2)


Para conservar el tejido del tiempo y permitir que la convolución funcione, Keras exige que empaquetes los datos en un bloque con volumen tridimensional. Matemáticamente, este tensor $\mathbf{X}$ pertenece al espacio $\mathbb{R}^{B \times T \times C}$

Si el X_train.shape arroja (340, 24, 2), significa que:
1. Dimensión 0: El Lote o Batch ($B = 340$)
- Qué es: Es la cantidad total de "ventanas" o escenarios independientes que logramos extraer de nuestro historial.
- Por qué lo exige la CNN: La red neuronal necesita saber exactamente cuántos ejemplos independientes tiene disponibles para calcular el error promedio y actualizar los pesos matemáticos antes del siguiente ciclo.
2. Dimensión 1: Pasos de Tiempo o Timesteps ($T = 24$)
- Qué es: El tamaño de la ventana (24 horas hacia atrás). Es el eje cronológico.- Por qué lo exige la CNN: Este es el único eje por donde se desliza la convolución 1D. El kernel de tamaño 3 se mueve hacia abajo por estas 24 filas buscando derivadas locales (ej. "el precio subió de la hora 5 a la 6 y a la 7").
3. Dimensión 2: Canales o Features ($C = 2$)
- Qué es: El número de variables que coexisten en un mismo instante de tiempo (Precio y Hora).
- Por qué lo exige la CNN: En imágenes (CNN 2D), estos son los canales de color RGB. En series de tiempo, son tus variables exógenas. Esto permite que el kernel multiplique y sume matemáticamente el Precio y la Hora simultáneamente, encontrando correlaciones cruzadas (ej. "Precio alto + Hora 18 = Patrón fuerte").

In [13]:
#@title Visualizar entrada de red neuronal para convolucionar


# B = Batch (Muestras), T = Timesteps (Tiempo), C = Channels (Variables)
B_vis = 30   # 3 Muestras independientes
T_vis = 24  # 10 Horas hacia atrás
C_vis = 2   # 2 Variables (Precio y Hora)

# meshgrid crea una cuadrícula 3D con todas las combinaciones posibles de X, Y, Z
x, y, z = np.meshgrid(np.arange(C_vis), np.arange(T_vis), np.arange(B_vis))


fig_tensor = go.Figure(data=[go.Scatter3d(
    x=x.flatten(),
    y=y.flatten(),
    z=z.flatten(),
    mode='markers',
    marker=dict(
        symbol='square', # Simulamos cubos o celdas de matriz
        size=12,         # Tamaño de cada celda
        color=y.flatten(), # Coloreamos en función del tiempo para ver el eje Y claramente
        colorscale='Viridis',
        opacity=0.9,
        line=dict(color='black', width=1) # Borde negro para separar las celdas
    ),
    # Etiqueta emergente al pasar el ratón (Hover)
    text=[f"Variable: {'Precio' if xi==0 else 'Hora'}<br>Paso de Tiempo: t-{T_vis-yi}<br>Muestra (Batch): {zi}"
          for xi, yi, zi in zip(x.flatten(), y.flatten(), z.flatten())],
    hoverinfo='text'
)])

fig_tensor.update_layout(
    title='<b>Anatomía del Tensor 3D para 1D-CNN en Keras</b><br><sup>Forma: [Batch_Size, Timesteps, Variables]</sup>',
    scene=dict(
        xaxis_title='Variables (Features)',
        yaxis_title='Eje Temporal (Timesteps)',
        zaxis_title='Muestras (Batch)',
        # Forzamos que el eje X solo muestre 0 y 1
        xaxis=dict(tickvals=[0, 1], ticktext=['0 (Precio)', '1 (Hora)']),
    ),
    margin=dict(l=0, r=0, b=0, t=60),
    template='plotly_white'
)

fig_tensor.show()

In [ ]:
#@title Diseñamos CNN con base a los tamaños de entrada

# Definimos el modelo y lo asignamos a la variable 'model'
model = Sequential([
    Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(WINDOW_SIZE, len(features))),
    MaxPooling1D(pool_size=2), # Numero de variables exogenas
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2), # Numero de variables exogenas,
    Flatten(), # IMPORTANTE!! Convierte el arreglo de 2 variables exogenas a uno lineal para capa Densa
    Dense(50, activation='relu'),
    Dense(1) #La de default es linear
])

model.compile(optimizer='adam', loss='mse')

# Entrenamos
model.fit(X_train, y_train, epochs=30, batch_size=32, validation_data=(X_test, y_test), verbose=1)


I0000 00:00:1777226785.217080    5060 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3644 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:09:00.0, compute capability: 7.5


Epoch 1/30


I0000 00:00:1777226786.448982    6283 service.cc:148] XLA service 0x7f15500074a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777226786.449006    6283 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 2060, Compute Capability 7.5
2026-04-26 12:06:26.474847: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777226786.576424    6283 cuda_dnn.cc:529] Loaded cuDNN version 90300


 1/10 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - loss: 0.0648

I0000 00:00:1777226787.576690    6283 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 128ms/step - loss: 0.0307 - val_loss: 0.0047
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0075 - val_loss: 0.0044
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0052 - val_loss: 0.0037
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0062 - val_loss: 0.0031
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0048 - val_loss: 0.0042
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0093 - val_loss: 0.0032
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0060 - val_loss: 0.0030
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0058 - val_loss: 0.0037
Epoch 9/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0049 - val_loss: 0.0031
Epoch 10/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0051 - val_loss: 0.0030
Epoch 11/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0056 - val_loss: 0.0029
Epoch 12/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0047 - val_loss: 0.003

In [15]:
#@title Desescalamos y graficamos predicciones en test

y_pred_scaled = model.predict(X_test)

# Invertimos escalado usando arreglo de ambas predicciones
def invertir_escalado(data_vector, scaler_obj):
    dummy_matrix = np.zeros((len(data_vector), 2))
    dummy_matrix[:, 0] = data_vector.flatten()
    return scaler_obj.inverse_transform(dummy_matrix)[:, 0]

y_pred_real = invertir_escalado(y_pred_scaled, scaler)
y_test_real = invertir_escalado(y_test, scaler)
y_train_real = invertir_escalado(y_train, scaler)

# Metricas
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
mape = mean_absolute_percentage_error(y_test_real, y_pred_real) * 100

# Ajuste de fechas para la gráfica
fechas_totales = df_precio['FechaHora'].values
fechas_train = fechas_totales[WINDOW_SIZE : WINDOW_SIZE + len(y_train)]
fechas_test = fechas_totales[WINDOW_SIZE + len(y_train) : ]

# Graficamos
fig_final = go.Figure()
fig_final.add_trace(go.Scatter(x=fechas_train, y=y_train_real, mode='lines', name='Entrenamiento', opacity=0.5))
fig_final.add_trace(go.Scatter(x=fechas_test, y=y_test_real, mode='lines', name='Realidad', line=dict(color='green')))
fig_final.add_trace(go.Scatter(x=fechas_test, y=y_pred_real, mode='lines', name='Predicción', line=dict(color='red', dash='dot')))

fig_final.update_layout(title=f'Pronóstico CNN - MAPE: {mape:.2f}% - RMSE ${rmse:.2f}MXN', template='plotly_white')
fig_final.show()

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step


In [16]:
#@title Visualizamos nuevamente prediciendo un horizonte mayor para validar


# Horizonte a predecir
horas_a_predecir = 48  #@param
ultima_ventana = test_data_scaled[-WINDOW_SIZE:].copy() # Iniciamos con las últimas 24h conocidas
proyecciones_scaled = []
fechas_futuras = []

# Referencia de tiempo
ultima_fecha_conocida = pd.to_datetime(fechas_test[-1])

print(f"Iniciando pronóstico recursivo para {horas_a_predecir} horas...")


for i in range(horas_a_predecir):
    # Preparamos el tensor de entrada (1, 24, 2)
    x_input = ultima_ventana.reshape(1, WINDOW_SIZE, 2)

    # Predecimos el siguiente precio (escalado)
    pred_precio_scaled = model.predict(x_input, verbose=0)[0][0]

    # Generamos la información temporal del futuro
    fecha_proyectada = ultima_fecha_conocida + timedelta(hours=i+1)
    hora_proyectada = fecha_proyectada.hour

    # Escalamos la hora usando el mismo scaler (requiere matriz de 2 cols)
    dummy_hora = np.zeros((1, 2))
    dummy_hora[0, 1] = hora_proyectada
    hora_proyectada_scaled = scaler.transform(dummy_hora)[0, 1]

    # Guardamos resultados
    proyecciones_scaled.append(pred_precio_scaled)
    fechas_futuras.append(fecha_proyectada)

    # Actualizamos la ventana: eliminamos el primer registro y añadimos la nueva predicción
    nuevo_registro = np.array([[pred_precio_scaled, hora_proyectada_scaled]])
    ultima_ventana = np.append(ultima_ventana[1:], nuevo_registro, axis=0)

# Regresamos a escala normal con la funcion que realizamos antes
y_futuro_real = invertir_escalado(np.array(proyecciones_scaled), scaler)

# Graficamos
fig_full = go.Figure()
# Train
fig_full.add_trace(go.Scatter(x=fechas_train, y=y_train_real, name='Entrenamiento', line=dict(width=1), opacity=0.4))
# Test
fig_full.add_trace(go.Scatter(x=fechas_test, y=y_test_real, name='Realidad (Test)', line=dict(color='green', width=2)))
# Predicción del modelo sobre el Test
fig_full.add_trace(go.Scatter(x=fechas_test, y=y_pred_real, name='Predicción Test', line=dict(color='red', dash='dot')))
# Prediccion del modelo a futuro desconocido
fig_full.add_trace(go.Scatter(x=fechas_futuras, y=y_futuro_real, name='Pronóstico 7 días', line=dict(color='orange', width=2)))

# Hoy
fig_full.add_vline(x=ultima_fecha_conocida, line_width=2, line_dash="dash", line_color="black")

fig_full.update_layout(
    title=f'Pronóstico Precios de Energía (Nodo: {nodo_objetivo})',
    xaxis_title='Fecha',
    yaxis_title='Precio (MXN/MWh)',
    template='plotly_white',
    hovermode='x unified'
)
fig_full.show()


Iniciando pronóstico recursivo para 48 horas...
